In [ ]:
pip install ultralytics opencv-python numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import cv2          
import numpy as np  
from ultralytics import YOLO  

In [3]:
model = YOLO("yolov8n-pose.pt")


In [ ]:
# =============================================================================
# Padel Analytics v5 — Google Colab Version
# =============================================================================
# SETUP INSTRUCTIONS (run these cells in order):
#
# CELL 1 — Install dependencies
# CELL 2 — Mount Google Drive
# CELL 3 — Run this script
#
# ─── CELL 1: Install ─────────────────────────────────────────────────────────
# !pip install ultralytics -q
#
# ─── CELL 2: Mount Drive ─────────────────────────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
#
# ─── CELL 3: Run script ──────────────────────────────────────────────────────
# Just run all cells below this point
# =============================================================================

# ── Colab environment check ───────────────────────────────────────────────────
import subprocess, sys, os

def in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

IS_COLAB = in_colab()

if IS_COLAB:
    print("✓ Running in Google Colab")
    # Auto-install if not present
    try:
        import ultralytics
    except ImportError:
        print("Installing ultralytics...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics", "-q"])
else:
    print("✓ Running locally")

# ─── IMPORTS ─────────────────────────────────────────────────────────────────

from ultralytics import YOLO
import cv2
import numpy as np
import pandas as pd
from collections import defaultdict, deque

# ─── CONFIGURATION ───────────────────────────────────────────────────────────

#
# After mounting Drive, your files are at:
#   /content/drive/MyDrive/<your folder>/<your file>
#
# Example if your video is in "My Drive > padel_analytics > input":
INPUT_VIDEO  = "/content/sample_croped_inputvedio.mp4"   # ← upload your video here
OUTPUT_VIDEO = "/content/output6_padel.mp4"
OUTPUT_CSV   = "/content/shot6_results.csv"

# Models — downloaded automatically by ultralytics on first run (no manual download needed)
DET_MODEL    = "yolov8m.pt"
POSE_MODEL   = "yolov8m-pose.pt"

MAX_PLAYERS          = 4
CONF_PERSON          = 0.45
CONF_BALL            = 0.25
CONF_RACKET          = 0.30

SHOT_COOLDOWN_FRAMES = 12
RALLY_RESET_FRAMES   = 60
WRIST_VELOCITY_MIN   = 8.0
ZONE_DEADBAND        = 0.08

BALL_TRAIL_LEN       = 20
BOUNCE_GROUND_FRAC   = 0.70

CLS_PERSON  = 0
CLS_BALL    = 32
CLS_RACKET  = 38

COLOR_PERSON  = (200, 200, 200)
COLOR_BALL    = (0,   255, 255)
COLOR_RACKET  = (255, 100, 0  )
COLOR_TRAIL   = (0,   200, 255)
COLOR_BOUNCE  = (0,   0,   255)

SHOT_COLORS = {
    "FOREHAND" : (0,   200, 0  ),
    "BACKHAND" : (255, 140, 0  ),
    "SMASH"    : (0,   0,   220),
    "SERVE"    : (255, 0,   200),
}

DIRECTION_COLORS = {
    "cross-court"   : (0,   220, 220),
    "down-the-line" : (220, 0,   220),
    "lob"           : (0,   180, 255),
}


# ─── HELPERS (unchanged from v5) ─────────────────────────────────────────────

def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba, bc  = a - b, c - b
    cos_a   = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return float(np.degrees(np.arccos(np.clip(cos_a, -1.0, 1.0))))


def get_arm_keypoints(kp):
    rs, re, rw = kp[6], kp[8], kp[10]
    if rs[0] > 0 or rs[1] > 0:
        return rs, re, rw
    ls, le, lw = kp[5], kp[7], kp[9]
    if ls[0] > 0 or ls[1] > 0:
        return ls, le, lw
    return None


def wrist_speed(curr_kp, prev_kp):
    if prev_kp is None:
        return 0.0
    rw_c = curr_kp[10] if curr_kp[10][0] > 0 else curr_kp[9]
    rw_p = prev_kp[10] if prev_kp[10][0]  > 0 else prev_kp[9]
    return float(np.linalg.norm(np.array(rw_c) - np.array(rw_p)))


def classify_shot(kp, is_rally_active):
    arm = get_arm_keypoints(kp)
    if arm is None:
        return None, is_rally_active
    rs, re, rw = arm
    arm_raised = rw[1] < rs[1]
    angle      = calculate_angle(rs, re, rw)
    if arm_raised:
        if not is_rally_active:
            return "SERVE", True
        elif angle < 90:
            return "SMASH", True
    if angle > 150:
        return "FOREHAND", is_rally_active
    return "BACKHAND", is_rally_active


def classify_direction(kp, cx, frame_w):
    arm = get_arm_keypoints(kp)
    if arm is None:
        return "unknown"
    rs, re, rw = arm
    head_y = kp[0][1]
    if rw[1] < head_y - 20 and rw[1] < rs[1]:
        return "lob"
    if rw[0] > 0 and abs(rw[0] - cx) > frame_w * 0.05:
        return "cross-court" if rw[0] < cx else "down-the-line"
    return "unknown"


def assign_zone_label(cx, cy, frame_w, frame_h, last_label, deadband=ZONE_DEADBAND):
    mid_x, mid_y = frame_w / 2, frame_h / 2
    if (abs(cx - mid_x) < frame_w * deadband or
            abs(cy - mid_y) < frame_h * deadband) and last_label:
        return last_label
    side  = "Left"  if cx < mid_x else "Right"
    depth = "Back"  if cy < mid_y else "Front"
    return f"{side}-{depth}"


def nearest_zone(cx, cy, frame_w, frame_h):
    side  = "Left"  if cx < frame_w / 2 else "Right"
    depth = "Back"  if cy < frame_h / 2 else "Front"
    return f"{side}-{depth}"


def draw_court_overlay(frame):
    h, w = frame.shape[:2]
    cv2.line(frame, (w // 2, 0),  (w // 2, h), (180, 180, 180), 1)
    cv2.line(frame, (0, h // 2),  (w, h // 2), (180, 180, 180), 1)


def draw_ball_trail(frame, trail):
    pts = list(trail)
    for i in range(1, len(pts)):
        if pts[i - 1] is None or pts[i] is None:
            continue
        alpha     = i / len(pts)
        thickness = max(1, int(4 * alpha))
        color     = tuple(int(c * alpha) for c in COLOR_TRAIL)
        cv2.line(frame, pts[i - 1], pts[i], color, thickness)


def draw_dashboard(frame, shot_counts, ball_events, frame_id, fps):
    h, w     = frame.shape[:2]
    panel_w  = 200
    panel_h  = 140
    x0, y0   = w - panel_w - 10, 10
    overlay  = frame.copy()
    cv2.rectangle(overlay, (x0, y0), (x0 + panel_w, y0 + panel_h), (20, 20, 20), -1)
    cv2.addWeighted(overlay, 0.55, frame, 0.45, 0, frame)
    cv2.putText(frame, "Live Analytics", (x0 + 6, y0 + 18),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (220, 220, 220), 1)
    for i, lbl in enumerate(["FOREHAND", "BACKHAND", "SMASH", "SERVE"]):
        color = SHOT_COLORS.get(lbl, (160, 160, 160))
        cv2.putText(frame, f"{lbl[:8]:<8} {shot_counts.get(lbl, 0):>4}",
                    (x0 + 6, y0 + 36 + i * 17),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.37, color, 1)
    cv2.putText(frame, f"Bounces  {ball_events['bounce']:>4}",
                (x0 + 6, y0 + 36 + 4 * 17),
                cv2.FONT_HERSHEY_SIMPLEX, 0.37, COLOR_BOUNCE, 1)
    cv2.putText(frame, f"T={frame_id/fps:.1f}s",
                (x0 + 6, y0 + panel_h - 6),
                cv2.FONT_HERSHEY_SIMPLEX, 0.34, (140, 140, 140), 1)


class BallTracker:
    def __init__(self, trail_len=BALL_TRAIL_LEN):
        self.trail         = deque(maxlen=trail_len)
        self.prev_cy       = None
        self.prev_dy       = 0
        self.bounce_frames = set()

    def update(self, cx, cy, frame_id, frame_h):
        if cx is None:
            self.trail.append(None)
            return False
        self.trail.append((cx, cy))
        is_bounce = False
        if self.prev_cy is not None:
            dy = cy - self.prev_cy
            if self.prev_dy > 2 and dy < -2 and cy > frame_h * BOUNCE_GROUND_FRAC:
                is_bounce = True
                self.bounce_frames.add(frame_id)
            self.prev_dy = dy
        self.prev_cy = cy
        return is_bounce

    def latest_pos(self):
        for pt in reversed(self.trail):
            if pt is not None:
                return pt
        return None


# ─── COLAB VIDEO PREVIEW HELPER ──────────────────────────────────────────────

def show_video_in_colab(video_path, max_frames=120):
    """
    Display a preview of the output video inline in Colab.
    Converts a small clip to a base64-encoded HTML5 video tag.
    Only runs inside Colab — silently skipped otherwise.
    """
    if not IS_COLAB:
        return

    try:
        from IPython.display import HTML, display
        import base64, tempfile

        cap_prev = cv2.VideoCapture(video_path)
        fps_prev = cap_prev.get(cv2.CAP_PROP_FPS) or 25.0

        tmp = tempfile.NamedTemporaryFile(suffix=".mp4", delete=False)
        tmp_path = tmp.name
        tmp.close()

        w = int(cap_prev.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap_prev.get(cv2.CAP_PROP_FRAME_HEIGHT))
        writer = cv2.VideoWriter(tmp_path,
                                 cv2.VideoWriter_fourcc(*"mp4v"),
                                 fps_prev, (w, h))
        count = 0
        while count < max_frames:
            ret, f = cap_prev.read()
            if not ret:
                break
            writer.write(f)
            count += 1

        cap_prev.release()
        writer.release()

        # Re-encode with ffmpeg for browser-compatible H.264
        h264_path = tmp_path.replace(".mp4", "_h264.mp4")
        os.system(f"ffmpeg -y -i {tmp_path} -vcodec libx264 -acodec aac {h264_path} -loglevel quiet")

        src_path = h264_path if os.path.exists(h264_path) else tmp_path
        with open(src_path, "rb") as vf:
            b64 = base64.b64encode(vf.read()).decode()

        display(HTML(f"""
            <p><b>Output video preview (first {max_frames} frames):</b></p>
            <video width="720" controls>
              <source src="data:video/mp4;base64,{b64}" type="video/mp4">
            </video>
        """))

    except Exception as e:
        print(f"Preview not available: {e}")
        print(f"Download the video directly from: {video_path}")


# ─── MAIN ────────────────────────────────────────────────────────────────────

def main():
    print(f"\nLoading models…")
    det_model  = YOLO(DET_MODEL)
    pose_model = YOLO(POSE_MODEL)
    print("✓ Models loaded")

    cap = cv2.VideoCapture(INPUT_VIDEO)
    if not cap.isOpened():
        raise FileNotFoundError(
            f"Cannot open: {INPUT_VIDEO}\n"
            "→ Check that your Google Drive is mounted and the path is correct."
        )

    fps    = cap.get(cv2.CAP_PROP_FPS) or 25.0
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"✓ Video opened: {width}×{height}  {fps:.0f}fps  {total} frames")

    out = cv2.VideoWriter(
        OUTPUT_VIDEO,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height),
    )

    results_list      = []
    frame_id          = 0
    is_rally_active   = False
    frames_since_shot = 0

    cooldown_remaining  = defaultdict(int)
    last_zone_label     = {}
    prev_keypoints      = {}
    unique_players      = set()
    shot_counts         = defaultdict(int)
    ball_events         = {"bounce": 0}
    ball_tracker        = BallTracker()
    bounce_flash_frames = 0

    print("\nProcessing video…")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_id += 1
        if frame_id % 100 == 0:
            pct = frame_id / total * 100 if total > 0 else 0
            print(f"  [{pct:5.1f}%] Frame {frame_id}/{total}  "
                  f"shots={sum(shot_counts.values())}  bounces={ball_events['bounce']}")

        draw_court_overlay(frame)

        # Rally reset
        frames_since_shot += 1
        if frames_since_shot > RALLY_RESET_FRAMES and is_rally_active:
            is_rally_active   = False
            frames_since_shot = 0

        for zone in list(cooldown_remaining):
            if cooldown_remaining[zone] > 0:
                cooldown_remaining[zone] -= 1

        # Detection
        det_results = det_model(frame, verbose=False)[0]

        boxes_all   = det_results.boxes.xyxy.cpu().numpy()  if det_results.boxes else np.empty((0, 4))
        classes_all = det_results.boxes.cls.cpu().numpy()   if det_results.boxes else np.array([])
        confs_all   = det_results.boxes.conf.cpu().numpy()  if det_results.boxes else np.array([])

        person_boxes      = []
        ball_detections   = []
        racket_detections = []

        for box, cls, conf in zip(boxes_all, classes_all, confs_all):
            cls = int(cls)
            if   cls == CLS_PERSON and conf > CONF_PERSON:
                person_boxes.append((box, conf))
            elif cls == CLS_BALL   and conf > CONF_BALL:
                ball_detections.append(box)
            elif cls == CLS_RACKET and conf > CONF_RACKET:
                racket_detections.append(box)

        # Ball
        ball_cx, ball_cy = None, None
        if ball_detections:
            best_box = max(ball_detections, key=lambda b: (b[2]-b[0]) * (b[3]-b[1]))
            bx1, by1, bx2, by2 = map(int, best_box)
            ball_cx = (bx1 + bx2) // 2
            ball_cy = (by1 + by2) // 2
            cv2.circle(frame, (ball_cx, ball_cy), 10, COLOR_BALL, 2)
            cv2.putText(frame, "BALL", (bx1, by1 - 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, COLOR_BALL, 1)

        is_bounce = ball_tracker.update(ball_cx, ball_cy, frame_id, height)
        if is_bounce:
            ball_events["bounce"] += 1
            bounce_flash_frames = 8
            bounce_zone = nearest_zone(ball_cx, ball_cy, width, height) if ball_cx else "unknown"
            results_list.append({
                "frame"         : frame_id,
                "timestamp_sec" : round(frame_id / fps, 2),
                "player"        : bounce_zone,
                "shot"          : "BOUNCE",
                "direction"     : "—",
            })

        draw_ball_trail(frame, ball_tracker.trail)

        if bounce_flash_frames > 0:
            pos = ball_tracker.latest_pos()
            if pos:
                cv2.circle(frame, pos, 20, COLOR_BOUNCE, 2)
                cv2.putText(frame, "BOUNCE!", (pos[0] - 30, pos[1] - 25),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, COLOR_BOUNCE, 2)
            bounce_flash_frames -= 1

        # Racket
        for rbox in racket_detections:
            rx1, ry1, rx2, ry2 = map(int, rbox)
            rcx, rcy = (rx1 + rx2) // 2, (ry1 + ry2) // 2
            zone = nearest_zone(rcx, rcy, width, height)
            cv2.rectangle(frame, (rx1, ry1), (rx2, ry2), COLOR_RACKET, 2)
            cv2.putText(frame, f"RACKET ({zone})", (rx1, ry1 - 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, COLOR_RACKET, 1)

        # Persons + pose
        person_boxes.sort(key=lambda x: x[1], reverse=True)
        person_boxes = person_boxes[:MAX_PLAYERS]
        players_this_frame = []

        for box, _ in person_boxes:
            x1, y1, x2, y2 = map(int, box)
            if x2 <= x1 or y2 <= y1:
                continue
            crop = frame[y1:y2, x1:x2]
            if crop.size == 0:
                continue

            pose_result = pose_model(crop, verbose=False)[0]
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2

            centroid_key = (cx // max(1, width // 2), cy // max(1, height // 2))
            last_lbl     = last_zone_label.get(centroid_key)
            zone         = assign_zone_label(cx, cy, width, height, last_lbl)
            last_zone_label[centroid_key] = zone

            shot      = None
            direction = "unknown"

            if pose_result.keypoints is not None:
                kp_all = pose_result.keypoints.xy.cpu().numpy()
                if len(kp_all) > 0:
                    kp    = kp_all[0]
                    speed = wrist_speed(kp, prev_keypoints.get(zone))
                    prev_keypoints[zone] = kp

                    for kx, ky in kp:
                        if kx > 0 and ky > 0:
                            cv2.circle(frame, (int(x1 + kx), int(y1 + ky)), 3, (0, 255, 0), -1)

                    if speed >= WRIST_VELOCITY_MIN and cooldown_remaining[zone] == 0:
                        shot, is_rally_active = classify_shot(kp, is_rally_active)
                        if shot is not None:
                            direction                = classify_direction(kp, cx, width)
                            cooldown_remaining[zone] = SHOT_COOLDOWN_FRAMES
                            frames_since_shot        = 0
                            shot_counts[shot]       += 1
                            unique_players.add(zone)
                            results_list.append({
                                "frame"         : frame_id,
                                "timestamp_sec" : round(frame_id / fps, 2),
                                "player"        : zone,
                                "shot"          : shot,
                                "direction"     : direction,
                            })

            players_this_frame.append((cx, cy, x1, y1, x2, y2, zone, shot, direction))

        for cx, cy, x1, y1, x2, y2, zone, shot, direction in players_this_frame:
            color = SHOT_COLORS.get(shot, COLOR_PERSON) if shot else COLOR_PERSON
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            text = f"{zone} | {shot}" if shot else zone
            (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.52, 2)
            cv2.rectangle(frame, (x1, y1 - th - 8), (x1 + tw + 4, y1), color, -1)
            cv2.putText(frame, text, (x1 + 2, y1 - 4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.52, (0, 0, 0), 2)
            if shot and direction != "unknown":
                dir_color = DIRECTION_COLORS.get(direction, (160, 160, 160))
                cv2.putText(frame, direction, (x1, y2 + 16),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.42, dir_color, 1)

        draw_dashboard(frame, shot_counts, ball_events, frame_id, fps)
        out.write(frame)

    cap.release()
    out.release()

    print(f"\n✓ Done! Processed {frame_id} frames.")
    print(f"✓ Output video → {OUTPUT_VIDEO}")

    # CSV
    df = pd.DataFrame(results_list,
                      columns=["frame", "timestamp_sec", "player", "shot", "direction"])

    summary_rows = []
    if not df.empty:
        for zone in sorted(df["player"].unique()):
            zdf    = df[df["player"] == zone]
            counts = zdf["shot"].value_counts().to_dict()
            summary_rows.append({
                "frame": "SUMMARY", "timestamp_sec": "—",
                "player": zone,
                "shot": " | ".join(f"{k}:{v}" for k, v in counts.items()),
                "direction": "—",
            })
    summary_rows.append({
        "frame": "SUMMARY", "timestamp_sec": "—",
        "player": f"TOTAL UNIQUE PLAYERS: {len(unique_players)}",
        "shot": ", ".join(sorted(unique_players)),
        "direction": f"Total bounces: {ball_events['bounce']}",
    })

    pd.concat([df, pd.DataFrame(summary_rows)], ignore_index=True).to_csv(OUTPUT_CSV, index=False)
    print(f"✓ CSV saved → {OUTPUT_CSV}")

    # Console summary
    shot_df = df[df["shot"] != "BOUNCE"]
    print("\n" + "═" * 54)
    print("  PADEL ANALYTICS — FINAL SUMMARY")
    print("═" * 54)
    print(f"  Total shots  : {len(shot_df)}")
    print(f"  Ball bounces : {ball_events['bounce']}")
    print(f"  Player zones : {len(unique_players)}")
    print(f"  Duration     : {frame_id/fps:.1f}s  ({frame_id} frames @ {fps:.0f}fps)")

    if not shot_df.empty:
        print("\n  Shot counts by player zone:\n")
        print(shot_df.groupby(["player", "shot"]).size().unstack(fill_value=0).to_string())
        print("\n  Overall shot totals:\n")
        for shot_type, count in shot_df["shot"].value_counts().items():
            pct = count / len(shot_df) * 100
            print(f"  {shot_type:<10} {count:>5}  {pct:5.1f}%  {'█' * int(pct/3)}")

    print("═" * 54)

    # Inline preview in Colab
    if IS_COLAB:
        print("\nGenerating inline preview…")
        show_video_in_colab(OUTPUT_VIDEO)

        # Also offer CSV download
        try:
            from google.colab import files
            print("\nDownloading CSV…")
            files.download(OUTPUT_CSV)
        except Exception:
            pass


# ─── ENTRY POINT ─────────────────────────────────────────────────────────────

main()   # In Colab, just call directly (no __main__ guard needed)

bonus_dashboard

In [ ]:
# =============================================================================
# Padel Analytics — Bonus Task Dashboard
# =============================================================================
# Run AFTER padel_analytics_v5_colab.py has produced shot5_results.csv
#
# What this script does:
#   1. Shot count analytics   — totals, per-player breakdown, forehand vs backhand ratio
#   2. Video overlay dashboard — bar chart + court heatmap burned into a summary video
#   3. Rule-based logic summary — shot direction counts, bounce zone map
#   4. HTML report             — self-contained file you can open in any browser
#
# Usage (Colab):
#   INPUT_CSV   = "/content/shot5_results.csv"
#   INPUT_VIDEO = "/content/sample_croped_inputvedio.mp4"
#   OUTPUT_VIDEO= "/content/summary_padel.mp4"
#   OUTPUT_HTML = "/content/padel_report.html"
# =============================================================================

import cv2
import numpy as np
import pandas as pd
import os
from collections import defaultdict

# ─── CONFIGURATION ───────────────────────────────────────────────────────────

INPUT_CSV    = "/content/shot_results2.csv"
INPUT_VIDEO  = "/content/sample_croped_inputvedio.mp4"
OUTPUT_VIDEO = "/content/final_output.mp4"
OUTPUT_HTML  = "/content/padel_report.html"

SHOT_COLORS_BGR = {
    "FOREHAND" : (0,   200, 0  ),
    "BACKHAND" : (255, 140, 0  ),
    "SMASH"    : (0,   0,   220),
    "SERVE"    : (255, 0,   200),
    "BOUNCE"   : (0,   0,   255),
}

ZONE_POSITIONS = {
    "Left-Back"   : (0.15, 0.25),
    "Left-Front"  : (0.15, 0.75),
    "Right-Back"  : (0.75, 0.25),
    "Right-Front" : (0.75, 0.75),
}


# ─── 1. SHOT COUNT ANALYTICS ─────────────────────────────────────────────────

def analyse_shots(df):
    """
    Returns a dict of analytics results from the shot CSV.
    """
    shot_df   = df[~df["shot"].isin(["BOUNCE", "SUMMARY"])].copy()
    bounce_df = df[df["shot"] == "BOUNCE"].copy()

    total_shots   = len(shot_df)
    total_bounces = len(bounce_df)

    # Overall shot type counts
    shot_totals = shot_df["shot"].value_counts().to_dict()

    # Per-player breakdown
    player_breakdown = (
        shot_df.groupby(["player", "shot"])
        .size()
        .unstack(fill_value=0)
    )

    # Forehand vs Backhand ratio per player
    fh_bh = {}
    for player in shot_df["player"].unique():
        pdf = shot_df[shot_df["player"] == player]
        fh  = len(pdf[pdf["shot"] == "FOREHAND"])
        bh  = len(pdf[pdf["shot"] == "BACKHAND"])
        fh_bh[player] = {
            "forehand"  : fh,
            "backhand"  : bh,
            "fh_ratio"  : round(fh / (fh + bh) * 100, 1) if (fh + bh) > 0 else 0,
        }

    # Direction breakdown
    dir_counts = {}
    if "direction" in shot_df.columns:
        dir_counts = shot_df["direction"].value_counts().to_dict()

    # Bounce zone distribution
    bounce_zones = {}
    if not bounce_df.empty and "player" in bounce_df.columns:
        bounce_zones = bounce_df["player"].value_counts().to_dict()

    # Shot timeline (shots per 5-second window)
    timeline = {}
    if "timestamp_sec" in shot_df.columns:
        shot_df["window"] = (pd.to_numeric(shot_df["timestamp_sec"], errors="coerce") // 5 * 5).astype("Int64")
        timeline = shot_df.groupby("window")["shot"].count().to_dict()

    return {
        "total_shots"      : total_shots,
        "total_bounces"    : total_bounces,
        "shot_totals"      : shot_totals,
        "player_breakdown" : player_breakdown,
        "fh_bh"            : fh_bh,
        "dir_counts"       : dir_counts,
        "bounce_zones"     : bounce_zones,
        "timeline"         : timeline,
    }


def print_analytics(stats):
    """Pretty-print analytics to console."""
    print("\n" + "═" * 60)
    print("  PADEL ANALYTICS — BONUS TASK SUMMARY")
    print("═" * 60)

    print(f"\n  Total shots detected : {stats['total_shots']}")
    print(f"  Total ball bounces   : {stats['total_bounces']}")

    print("\n  ── Overall shot type counts ─────────────────────────────")
    for shot, count in sorted(stats["shot_totals"].items(), key=lambda x: -x[1]):
        pct = count / stats["total_shots"] * 100 if stats["total_shots"] else 0
        bar = "█" * int(pct / 2)
        print(f"  {shot:<12} {count:>5}  {pct:5.1f}%  {bar}")

    print("\n  ── Forehand vs Backhand ratio per player ────────────────")
    for player, d in stats["fh_bh"].items():
        fh_bar = "▓" * int(d["fh_ratio"] / 5)
        bh_bar = "░" * int((100 - d["fh_ratio"]) / 5)
        print(f"  {player:<14}  FH {d['forehand']:>4}  BH {d['backhand']:>4}  "
              f"FH%={d['fh_ratio']:>5.1f}%  {fh_bar}{bh_bar}")

    if stats["dir_counts"]:
        print("\n  ── Shot direction (rule-based) ───────────────────────────")
        for direction, count in sorted(stats["dir_counts"].items(), key=lambda x: -x[1]):
            print(f"  {direction:<20} {count:>5}")

    if stats["bounce_zones"]:
        print("\n  ── Bounce zone distribution ─────────────────────────────")
        for zone, count in sorted(stats["bounce_zones"].items(), key=lambda x: -x[1]):
            print(f"  {zone:<16} {count:>4} bounces")

    print("\n" + "═" * 60)


# ─── 2. VIDEO OVERLAY DASHBOARD ──────────────────────────────────────────────

def draw_bar_chart(frame, shot_totals, x0, y0, w=220, h=150):
    """
    Draw a mini bar chart of shot totals onto the frame.
    """
    if not shot_totals:
        return

    shots  = [s for s in ["FOREHAND", "BACKHAND", "SMASH", "SERVE"] if s in shot_totals]
    counts = [shot_totals[s] for s in shots]
    max_c  = max(counts) if counts else 1

    # Background panel
    overlay = frame.copy()
    cv2.rectangle(overlay, (x0, y0), (x0 + w, y0 + h + 30), (15, 15, 15), -1)
    cv2.addWeighted(overlay, 0.65, frame, 0.35, 0, frame)

    cv2.putText(frame, "Shot breakdown", (x0 + 6, y0 + 16),
                cv2.FONT_HERSHEY_SIMPLEX, 0.42, (200, 200, 200), 1)

    bar_w    = (w - 20) // max(len(shots), 1)
    chart_y0 = y0 + 25
    chart_h  = h - 10

    for i, (shot, count) in enumerate(zip(shots, counts)):
        bx     = x0 + 10 + i * bar_w
        bar_h  = int((count / max_c) * (chart_h - 20))
        color  = SHOT_COLORS_BGR.get(shot, (150, 150, 150))

        # Bar
        cv2.rectangle(frame,
                      (bx, chart_y0 + chart_h - bar_h),
                      (bx + bar_w - 4, chart_y0 + chart_h - 5),
                      color, -1)

        # Count label
        cv2.putText(frame, str(count),
                    (bx + 2, chart_y0 + chart_h - bar_h - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.32, (220, 220, 220), 1)

        # Shot label (first 2 chars)
        cv2.putText(frame, shot[:2],
                    (bx + 2, chart_y0 + chart_h + 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.32, color, 1)


def draw_court_heatmap(frame, fh_bh, x0, y0, w=220, h=180):
    """
    Draw a top-down court diagram with FH% shown in each zone.
    """
    overlay = frame.copy()
    cv2.rectangle(overlay, (x0, y0), (x0 + w, y0 + h), (15, 15, 15), -1)
    cv2.addWeighted(overlay, 0.65, frame, 0.35, 0, frame)

    cv2.putText(frame, "Forehand % by zone", (x0 + 6, y0 + 14),
                cv2.FONT_HERSHEY_SIMPLEX, 0.38, (200, 200, 200), 1)

    # Court outline
    cx0, cy0 = x0 + 10, y0 + 22
    cw,  ch  = w - 20, h - 30
    cv2.rectangle(frame, (cx0, cy0), (cx0 + cw, cy0 + ch), (120, 120, 120), 1)
    # Midlines
    cv2.line(frame, (cx0 + cw // 2, cy0), (cx0 + cw // 2, cy0 + ch), (80, 80, 80), 1)
    cv2.line(frame, (cx0, cy0 + ch // 2), (cx0 + cw, cy0 + ch // 2), (80, 80, 80), 1)

    zone_rects = {
        "Left-Back"   : (cx0,           cy0,           cx0 + cw//2, cy0 + ch//2),
        "Right-Back"  : (cx0 + cw//2,   cy0,           cx0 + cw,   cy0 + ch//2),
        "Left-Front"  : (cx0,           cy0 + ch//2,   cx0 + cw//2, cy0 + ch),
        "Right-Front" : (cx0 + cw//2,   cy0 + ch//2,   cx0 + cw,   cy0 + ch),
    }

    for zone, rect in zone_rects.items():
        zx0, zy0, zx1, zy1 = rect
        d = fh_bh.get(zone, {})
        fh_pct = d.get("fh_ratio", 0)

        # Colour tint by FH% (green = forehand heavy, blue = backhand heavy)
        g = int(fh_pct * 1.5)
        b = int((100 - fh_pct) * 1.5)
        fill_overlay = frame.copy()
        cv2.rectangle(fill_overlay, (zx0+1, zy0+1), (zx1-1, zy1-1), (b, g, 0), -1)
        cv2.addWeighted(fill_overlay, 0.25, frame, 0.75, 0, frame)

        mid_x = (zx0 + zx1) // 2 - 20
        mid_y = (zy0 + zy1) // 2

        cv2.putText(frame, zone.split("-")[0][:1] + zone.split("-")[1][:1],
                    (mid_x, mid_y - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.32, (180, 180, 180), 1)
        cv2.putText(frame, f"FH {fh_pct:.0f}%",
                    (mid_x - 4, mid_y + 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.32, (220, 220, 220), 1)


def draw_direction_pie_text(frame, dir_counts, x0, y0, w=220):
    """
    Draw a simple text-based direction summary (no pie — cv2 has no arc).
    """
    overlay = frame.copy()
    cv2.rectangle(overlay, (x0, y0), (x0 + w, y0 + 80), (15, 15, 15), -1)
    cv2.addWeighted(overlay, 0.65, frame, 0.35, 0, frame)

    cv2.putText(frame, "Shot direction", (x0 + 6, y0 + 14),
                cv2.FONT_HERSHEY_SIMPLEX, 0.38, (200, 200, 200), 1)

    total = sum(dir_counts.values()) or 1
    dir_colors = {
        "cross-court"   : (0,   220, 220),
        "down-the-line" : (220, 0,   220),
        "lob"           : (0,   180, 255),
        "unknown"       : (100, 100, 100),
    }
    y = y0 + 30
    for direction, count in sorted(dir_counts.items(), key=lambda x: -x[1]):
        if direction == "unknown":
            continue
        pct   = count / total * 100
        color = dir_colors.get(direction, (150, 150, 150))
        bar   = int(pct / 100 * (w - 80))
        cv2.rectangle(frame, (x0 + 10, y - 8), (x0 + 10 + bar, y - 1), color, -1)
        cv2.putText(frame, f"{direction[:12]} {pct:.0f}%",
                    (x0 + 10, y + 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.32, color, 1)
        y += 20


def produce_summary_video(stats, input_video, output_video):
    """
    Reads the original video and burns the analytics dashboard onto every frame.
    """
    cap = cv2.VideoCapture(input_video)
    if not cap.isOpened():
        print(f"  ⚠ Cannot open video: {input_video} — skipping video output")
        return

    fps    = cap.get(cv2.CAP_PROP_FPS) or 25.0
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    out = cv2.VideoWriter(output_video,
                          cv2.VideoWriter_fourcc(*"mp4v"),
                          fps, (width, height))

    print(f"  Rendering summary video ({total} frames)…")
    frame_id = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_id += 1

        # ── Left panel: bar chart ─────────────────────────────────────────
        draw_bar_chart(frame, stats["shot_totals"], x0=10, y0=10)

        # ── Left panel below: direction ───────────────────────────────────
        if stats["dir_counts"]:
            draw_direction_pie_text(frame, stats["dir_counts"], x0=10, y0=175)

        # ── Right panel: court heatmap ────────────────────────────────────
        draw_court_heatmap(frame, stats["fh_bh"],
                           x0=width - 235, y0=10)

        # ── Bottom strip: shot totals text ────────────────────────────────
        summary_text = (
            f"Shots: {stats['total_shots']}   "
            f"FH: {stats['shot_totals'].get('FOREHAND', 0)}   "
            f"BH: {stats['shot_totals'].get('BACKHAND', 0)}   "
            f"Smash: {stats['shot_totals'].get('SMASH', 0)}   "
            f"Bounces: {stats['total_bounces']}"
        )
        overlay = frame.copy()
        cv2.rectangle(overlay, (0, height - 28), (width, height), (10, 10, 10), -1)
        cv2.addWeighted(overlay, 0.7, frame, 0.3, 0, frame)
        cv2.putText(frame, summary_text, (12, height - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.48, (220, 220, 220), 1)

        out.write(frame)

    cap.release()
    out.release()
    print(f"  ✓ Summary video saved → {output_video}")


# ─── 3. HTML REPORT ──────────────────────────────────────────────────────────

def produce_html_report(stats, output_html):
    """
    Generate a self-contained HTML report with Chart.js visualisations.
    """

    # Build per-player table rows
    player_rows = ""
    for player, d in stats["fh_bh"].items():
        total = d["forehand"] + d["backhand"]
        smash = stats["shot_totals"].get("SMASH", 0)
        player_rows += f"""
        <tr>
          <td>{player}</td>
          <td>{d['forehand']}</td>
          <td>{d['backhand']}</td>
          <td>{d['fh_ratio']}%</td>
          <td>{total}</td>
        </tr>"""

    # Shot totals for Chart.js
    shot_labels  = list(stats["shot_totals"].keys())
    shot_values  = list(stats["shot_totals"].values())
    shot_colors  = ["#3a9e6c", "#e68c00", "#c0392b", "#cc00aa"]

    # Direction data
    dir_labels = [k for k in stats["dir_counts"] if k != "unknown"]
    dir_values = [stats["dir_counts"][k] for k in dir_labels]
    dir_colors = ["#00dcdc", "#dc00dc", "#00b4ff"]

    # Timeline data
    tl_labels = [str(int(k)) + "s" for k in sorted(stats["timeline"].keys())]
    tl_values = [stats["timeline"][k] for k in sorted(stats["timeline"].keys())]

    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Padel Analytics Report</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>
<style>
  *, *::before, *::after {{ box-sizing: border-box; margin: 0; padding: 0; }}
  body {{ font-family: system-ui, sans-serif; background: #f5f5f3; color: #1a1a1a; padding: 2rem; }}
  h1 {{ font-size: 22px; font-weight: 500; margin-bottom: 0.25rem; }}
  h2 {{ font-size: 15px; font-weight: 500; margin-bottom: 1rem; color: #444; }}
  .subtitle {{ font-size: 13px; color: #888; margin-bottom: 2rem; }}
  .metrics {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(130px, 1fr)); gap: 12px; margin-bottom: 2rem; }}
  .metric {{ background: #fff; border-radius: 10px; padding: 1rem; border: 0.5px solid #e0e0e0; }}
  .metric-label {{ font-size: 12px; color: #888; margin-bottom: 4px; }}
  .metric-value {{ font-size: 26px; font-weight: 500; }}
  .charts {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(280px, 1fr)); gap: 1.5rem; margin-bottom: 2rem; }}
  .card {{ background: #fff; border-radius: 10px; border: 0.5px solid #e0e0e0; padding: 1.25rem; }}
  .chart-wrap {{ position: relative; width: 100%; height: 220px; }}
  table {{ width: 100%; border-collapse: collapse; font-size: 13px; }}
  th {{ text-align: left; padding: 8px 12px; background: #f5f5f3; color: #888; font-weight: 500; border-bottom: 0.5px solid #e0e0e0; }}
  td {{ padding: 8px 12px; border-bottom: 0.5px solid #f0f0f0; }}
  tr:last-child td {{ border-bottom: none; }}
  .legend {{ display: flex; flex-wrap: wrap; gap: 14px; margin-bottom: 8px; font-size: 12px; color: #666; }}
  .legend span {{ display: flex; align-items: center; gap: 5px; }}
  .dot {{ width: 10px; height: 10px; border-radius: 2px; }}
  .full-width {{ grid-column: 1 / -1; }}
</style>
</head>
<body>
<h1>Padel Analytics Report</h1>
<p class="subtitle">Generated from shot detection pipeline · {stats['total_shots']} shots · {stats['total_bounces']} bounces</p>

<div class="metrics">
  <div class="metric"><div class="metric-label">Total shots</div><div class="metric-value">{stats['total_shots']}</div></div>
  <div class="metric"><div class="metric-label">Forehand</div><div class="metric-value" style="color:#3a9e6c">{stats['shot_totals'].get('FOREHAND',0)}</div></div>
  <div class="metric"><div class="metric-label">Backhand</div><div class="metric-value" style="color:#e68c00">{stats['shot_totals'].get('BACKHAND',0)}</div></div>
  <div class="metric"><div class="metric-label">Smash</div><div class="metric-value" style="color:#c0392b">{stats['shot_totals'].get('SMASH',0)}</div></div>
  <div class="metric"><div class="metric-label">Serve</div><div class="metric-value" style="color:#cc00aa">{stats['shot_totals'].get('SERVE',0)}</div></div>
  <div class="metric"><div class="metric-label">Ball bounces</div><div class="metric-value">{stats['total_bounces']}</div></div>
</div>

<div class="charts">

  <div class="card">
    <h2>Shot type breakdown</h2>
    <div class="legend">
      <span><span class="dot" style="background:#3a9e6c"></span>Forehand</span>
      <span><span class="dot" style="background:#e68c00"></span>Backhand</span>
      <span><span class="dot" style="background:#c0392b"></span>Smash</span>
      <span><span class="dot" style="background:#cc00aa"></span>Serve</span>
    </div>
    <div class="chart-wrap"><canvas id="shotChart" role="img" aria-label="Bar chart of total shots by type: Forehand {stats['shot_totals'].get('FOREHAND',0)}, Backhand {stats['shot_totals'].get('BACKHAND',0)}, Smash {stats['shot_totals'].get('SMASH',0)}, Serve {stats['shot_totals'].get('SERVE',0)}"></canvas></div>
  </div>

  <div class="card">
    <h2>Shot direction (rule-based)</h2>
    <div class="legend">
      <span><span class="dot" style="background:#00dcdc"></span>Cross-court</span>
      <span><span class="dot" style="background:#dc00dc"></span>Down-the-line</span>
      <span><span class="dot" style="background:#00b4ff"></span>Lob</span>
    </div>
    <div class="chart-wrap"><canvas id="dirChart" role="img" aria-label="Doughnut chart of shot directions"></canvas></div>
  </div>

  <div class="card full-width">
    <h2>Shot activity over time</h2>
    <div class="chart-wrap" style="height:160px"><canvas id="timelineChart" role="img" aria-label="Line chart of shot counts per 5-second window over video duration"></canvas></div>
  </div>

</div>

<div class="card" style="margin-bottom: 2rem;">
  <h2>Forehand vs backhand per player zone</h2>
  <table>
    <thead><tr><th>Player zone</th><th>Forehand</th><th>Backhand</th><th>FH %</th><th>Total</th></tr></thead>
    <tbody>{player_rows}</tbody>
  </table>
</div>

<script>
new Chart(document.getElementById('shotChart'), {{
  type: 'bar',
  data: {{
    labels: {shot_labels},
    datasets: [{{
      data: {shot_values},
      backgroundColor: {shot_colors},
      borderWidth: 0,
      borderRadius: 4,
    }}]
  }},
  options: {{
    responsive: true, maintainAspectRatio: false,
    plugins: {{ legend: {{ display: false }} }},
    scales: {{
      x: {{ grid: {{ display: false }}, ticks: {{ color: '#888' }} }},
      y: {{ grid: {{ color: 'rgba(0,0,0,0.06)' }}, ticks: {{ color: '#888' }} }}
    }}
  }}
}});

new Chart(document.getElementById('dirChart'), {{
  type: 'doughnut',
  data: {{
    labels: {dir_labels},
    datasets: [{{ data: {dir_values}, backgroundColor: {dir_colors}, borderWidth: 0 }}]
  }},
  options: {{
    responsive: true, maintainAspectRatio: false,
    plugins: {{ legend: {{ display: false }} }},
    cutout: '60%'
  }}
}});

new Chart(document.getElementById('timelineChart'), {{
  type: 'line',
  data: {{
    labels: {tl_labels},
    datasets: [{{
      label: 'Shots per 5s',
      data: {tl_values},
      borderColor: '#3266ad',
      backgroundColor: 'rgba(50,102,173,0.1)',
      fill: true,
      tension: 0.4,
      pointRadius: 2,
      borderWidth: 2,
    }}]
  }},
  options: {{
    responsive: true, maintainAspectRatio: false,
    plugins: {{ legend: {{ display: false }} }},
    scales: {{
      x: {{ grid: {{ display: false }}, ticks: {{ color: '#888', maxTicksLimit: 12 }} }},
      y: {{ grid: {{ color: 'rgba(0,0,0,0.06)' }}, ticks: {{ color: '#888' }} }}
    }}
  }}
}});
</script>
</body>
</html>"""

    with open(output_html, "w") as f:
        f.write(html)

    print(f"  ✓ HTML report saved → {output_html}")


# ─── MAIN ────────────────────────────────────────────────────────────────────

def main():
    print("Loading CSV…")
    try:
        df = pd.read_csv(INPUT_CSV)
    except FileNotFoundError:
        raise FileNotFoundError(
            f"CSV not found: {INPUT_CSV}\n"
            "Run padel_analytics_v5_colab.py first to generate it."
        )

    print(f"✓ Loaded {len(df)} rows from {INPUT_CSV}")

    # 1. Analytics
    print("\n── Shot count analytics ─────────────────────────────────────")
    stats = analyse_shots(df)
    print_analytics(stats)

    # 2. Summary video
    print("\n── Video overlay dashboard ──────────────────────────────────")
    produce_summary_video(stats, INPUT_VIDEO, OUTPUT_VIDEO)

    # 3. HTML report
    print("\n── HTML report ──────────────────────────────────────────────")
    produce_html_report(stats, OUTPUT_HTML)

    # 4. Colab downloads
    try:
        from google.colab import files
        print("\nDownloading outputs…")
        if os.path.exists(OUTPUT_VIDEO):
            files.download(OUTPUT_VIDEO)
        if os.path.exists(OUTPUT_HTML):
            files.download(OUTPUT_HTML)
    except ImportError:
        pass

    print("\n✓ Bonus task complete.")


main()